# Chapter 6 — Algorithm Chains and Pipelines

Pipeline menyatukan preprocessing dan model agar workflow lebih rapi dan aman dari data leakage.

In [1]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [2]:
from sklearn.datasets import load_breast_cancer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectPercentile, f_classif
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
cancer = load_breast_cancer(); X, y = cancer.data, cancer.target
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=RANDOM_STATE)

## 1. Pipeline Dasar

In [3]:
pipe = Pipeline([('scaler', StandardScaler()), ('svm', SVC())])
pipe.fit(X_train, y_train)
print('Akurasi test pipeline:', round(pipe.score(X_test,y_test),4))

Akurasi test pipeline: 0.979


## 2. Grid Search pada Pipeline

In [4]:
param_grid = {'svm__C':[0.1,1,10,100], 'svm__gamma':[0.001,0.01,0.1]}
grid = GridSearchCV(pipe, param_grid, cv=5)
grid.fit(X_train, y_train)
print('Best params:', grid.best_params_)
print('Best CV:', round(grid.best_score_,4))
print('Test score:', round(grid.score(X_test,y_test),4))

Best params: {'svm__C': 10, 'svm__gamma': 0.001}
Best CV: 0.9765
Test score: 0.979


## 3. Pipeline dengan Feature Selection

In [5]:
pipe_select = Pipeline([('scaler', StandardScaler()), ('select', SelectPercentile(f_classif)), ('logreg', LogisticRegression(max_iter=5000))])
param_grid = {'select__percentile':[30,50,70,100], 'logreg__C':[0.1,1,10]}
grid_select = GridSearchCV(pipe_select, param_grid, cv=5)
grid_select.fit(X_train, y_train)
print('Best params:', grid_select.best_params_)
print('Test score:', round(grid_select.score(X_test,y_test),4))

Best params: {'logreg__C': 0.1, 'select__percentile': 100}
Test score: 0.979


## 4. make_pipeline dan Perbandingan Model

In [6]:
models = {
    'LogReg': make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)),
    'SVM': make_pipeline(StandardScaler(), SVC(C=10, gamma=0.01)),
    'RandomForest': RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)
}
for name, model in models.items():
    cv = cross_val_score(model, X_train, y_train, cv=5)
    model.fit(X_train, y_train)
    print(f'{name:12s} | CV={cv.mean():.3f} | Test={model.score(X_test,y_test):.3f}')

LogReg       | CV=0.969 | Test=0.986
SVM          | CV=0.962 | Test=0.979


RandomForest | CV=0.960 | Test=0.958


## Kesimpulan

Pipeline memastikan preprocessing diterapkan dengan benar pada data train dan test, serta aman digunakan dalam GridSearchCV.